# Calculating Power-law scaling in fluctuations of information

In [1]:
import gc
import os
import numpy as np
import pandas as pd

from scipy.stats import pearsonr as r2
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = 'complexity-matching'
CORRELATION_VAR = 'AVAR'

In [3]:
DATA_PATH = '../data/ckpts/rosen'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
def corpus_name(x):
    result = str(x)

    if '-DISPEL-' in result:
        result = 'DISPEL'

    if result.startswith('GCSAusE'):
        result = 'GCSAusE'

    if result.startswith('se'):
        result = 'CORAAL'

    if result.startswith('call'):
        result = x.split('-')[0]

    if result.startswith('MICASE'):
        result = 'MICASE'

    if result.startswith('instruction'):
        result = x.split('-')[1]

    if result.startswith('CABNC'):
        result = 'CABNC'

    if result.startswith('SBCSAE'):
        result = 'SBCSAE'

    if result.startswith('SCoSE'):
        result = 'SCoSE'

    if result.startswith('candor'):
        result = 'CANDOR'

    if result.startswith('grace'):
        result = 'Miao-fNIRS'

    return result

In [6]:
fs = [f for f in grab_all_files(DATA_PATH) if (not f.split('/')[-1].startswith('.'))]
len(fs)

1669

## Processing individual datasets

In [7]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [8]:
COEF_VAR = 'AVAR'

In [9]:
param_list = 'time_delta'

In [10]:
df_scale, df_regression, k_docs, pct = [], [], dict(), 1.

In [11]:
for f in tqdm(fs):

    # print(f.split('/')[-1])

    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()

    df['self'] = (df['x_speaker'] == df['y_speaker'])

    df['it_count'] = df.groupby(by=['corpus', 'conversation_id', 'x_speaker', 'x_turn_id', 'self']).cumcount()

    groups = df.groupby(by=['corpus', 'conversation_id', 'x_speaker', 'x_turn_id'])


    for labels, dfi in groups:

        CORPUS_NAME = corpus_name(labels[0])

        k = pd.merge(
            left=dfi[['it_count']+[COEF_VAR]].loc[dfi['self'] & (~dfi[COEF_VAR].isna())], left_on=['it_count'],
            right=dfi[['it_count']+[COEF_VAR]].loc[~dfi['self'] & (~dfi[COEF_VAR].isna())], right_on=['it_count'],
            how='inner'
        )

        if len(k) > 1:

            CM = np.log((np.exp(k[COEF_VAR+'_x'].values) - np.exp(k[COEF_VAR+'_y'].values)).__abs__())

            df_regression += [
                    {
                        'corpus': CORPUS_NAME,
                        'cid': labels[0],
                        'speaker': labels[1],
                        # 'self': labels[2],
                        'CM': CM.sum(),
                        'k': len(k)
                    }
                ]





    #
    #
    #
    # df['x_turn_id_'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    # df[param_list] = (df['y_turn_id'] - df['x_turn_id_']) + 1
    #
    # df = df.loc[
    #     (~df[COEF_VAR].isna())
    #     & (~df[param_list].isna())
    # ]
    # len(df)
    #
    # for corpus in df['corpus'].unique():
    #
    #     CORPUS_NAME = corpus_name(corpus)
    #
    #     if CORPUS_NAME not in k_docs.keys():
    #         k_docs[CORPUS_NAME] = {
    #             True: 0,
    #             False: 0
    #         }
    #
    #     sub = df.loc[df['corpus'].isin([corpus])]
    #
    #     keep_data = sub['x_turn_id'].unique()
    #
    #     keep_data = np.random.choice(keep_data, size=(int(np.ceil(pct*len(keep_data)),),), replace=False)
    #
    #     sub = sub.loc[sub['x_turn_id'].isin(keep_data)]
    #     gc.collect()
    #
    #     groups = sub.groupby(by=['conversation_id', 'x_speaker'])
    #     for labels, dfi in groups:
    #
    #         if len(dfi):
    #             k_docs[CORPUS_NAME][True] += dfi['self'].sum()
    #             k_docs[CORPUS_NAME][False] += (~dfi['self']).sum()
    #
    #             dfi['it_count'] = dfi.groupby(by=['x_turn_id']).cumcount()
    #
    #             dfi = pd.merge(
    #                 left=dfi.loc[dfi['self']], left_on=['x_turn_id', 'it_count'],
    #                 right=dfi[['x_turn_id', 'it_count', 'I']].loc[~dfi['self']], right_on=['x_turn_id', 'it_count'],
    #                 how='left'
    #             )
    #             print(len(dfi), dfi.isna().sum())

                # if len(dfi) >= 2:
                #
                #     ccoef, _ = r2(dfi[COEF_VAR+'_x'].astype(float).values, dfi[COEF_VAR+'_y'].astype(float).values)
                #
                #     # b,a = np.polyfit(np.log(dfi[param_list].sample(frac=1.).values), np.log(dfi['AVAR'].sample(frac=1.).values + 1e-9), 1)
                #
                #     df_regression += [
                #         {
                #             'corpus': CORPUS_NAME,
                #             'cid': labels[0],
                #             'speaker': labels[1],
                #             'self': labels[2],
                #             'r': ccoef,
                #             'k': len(dfi)
                #         }
                #     ]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1669 [00:00<?, ?it/s]

In [12]:
df_regression.shape

(1243651, 5)

In [13]:
np.isinf(df_regression['CM']).sum()

np.int64(4014)

In [20]:
df_regression.to_parquet('complexity-match-data.parquet', engine='fastparquet', compression='snappy')

In [11]:
# df_regression.isna().sum()

In [12]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [13]:
# if not os.path.exists(os.path.join(OUTPUTS_PATH,'all.csv')):
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all.csv'),
#         index=False,
#         encoding='utf-8'
#     )
#
# else:
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all.csv'),
#         index=False,
#         header=False,
#         encoding='utf-8',
#         mode='a'
#     )

In [15]:
sel = ~np.isinf(df_regression['CM'])

In [16]:
dfr = df_regression.loc[sel]

## Aggregate statistics

In [19]:
mu, sd, k = dfr['CM'].mean(), dfr['CM'].std(), len(dfr)
stat = mu / (sd/np.sqrt(k))
stat, sd, mu, k, t.sf(stat.__abs__(), k-1)

(np.float64(-1306.5798021137534),
 np.float64(796.6858911494654),
 np.float64(-934.9229080953693),
 1239637,
 np.float64(0.0))

## Corpus level analysis of results

In [ ]:
sel = ~np.isinf(df_regression['CM'])

In [42]:
split_by = ['corpus', 'self']
# split_by = ['self']

In [12]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

In [43]:
k_docs = df_regression[split_by].value_counts().reset_index()
k_docs
# k_docs = k_docs.rename(columns={'count'})
# df0

KeyError: "None of [Index(['self'], dtype='object')] are in the [columns]"

In [ ]:
df0['k'] = df0

In [ ]:
df0['k'] = [sum([k_docs[corpus][s] for corpus in k_docs.keys()]) for s in df0[split_by].values]

In [13]:
df0['se'] = df0['std'] / np.sqrt(df0['k'])
df0['t'] = df0['b'] / df0['se']

In [14]:
df0.head(n=20)

,corpus,self,b,std,k,se,t
0,null,False,-0.032884,0.918894,12604,0.008185,-4.017643
1,null,True,-0.062977,1.133411,11290,0.010667,-5.903969


In [15]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)